In [1]:
import scanpy as sc
import pandas as pd
import anndata
import re
from tqdm import tqdm

In [2]:
# Load all DMSOs

In [3]:
file0 = '/lustre/groups/ml01/workspace/manuel.gander/data/sub_adatas/bulked/'

In [4]:
# and the single files

In [5]:
adatas = []
for plate in tqdm(range(1,15)):
    plate = str(plate)
    adata = sc.read_h5ad(file0+plate+'.h5ad')
    adata.obs['plate'] = plate
    adatas.append(adata)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [00:06<00:00,  2.30it/s]


In [17]:
adata = anndata.concat(adatas)

In [18]:
adata.write(file0+'all_comb.h5ad', compression='gzip')

In [12]:
adata = sc.read_h5ad(file0+'all_comb.h5ad')

In [6]:
import os

path = '/lustre/groups/ml01/workspace/manuel.gander/data/prelimma/mD3/'
os.makedirs(path, exist_ok=True)

In [16]:
file1 = '/lustre/groups/ml01/workspace/manuel.gander/data/prelimma/mD3/'

In [17]:
celllines = sorted(set(adata.obs.cell_name))

In [ ]:
for cellline in celllines:
    ads = adata[adata.obs['cell_name']==cellline].copy()
    cellline = re.sub(r'[^a-zA-Z0-9]', '', cellline)
    print(cellline)
    
    vc = ads.obs[['drugname_drugconc', 'n_cells']].groupby('drugname_drugconc').sum()
    c_kept = vc[vc.n_cells>100].index.values
    adss = ads[ads.obs.drugname_drugconc.isin(c_kept)].copy()
    
    a0 = ads.obs['plate'].astype('str').values
    a1 = ads.obs['drugname_drugconc'].astype('str').values
    
    ads.obs['drugname_drugconc'] = [a+'_'+b if b!="[('DMSO_TF', 0.0, 'uM')]" else b for a,b in zip(a0,a1)]
    
    pd.DataFrame(adss.X).to_csv(file1+cellline+"_X.csv", index=False)
    adss.var.to_csv(file1+cellline+"_var.csv")
    adss.obs.to_csv(file1+cellline+"_obs.csv")

A172
